# E-Commerce Clothing Classifier

End-to-end image classification for fashion retail. We train and compare two models on Fashion-MNIST:

1. **Simple CNN** — lightweight, fast, ~225K params
2. **MobileNetV2 Transfer Learning** — higher capacity, ImageNet features

Then we productionise: data augmentation, multi-label attribute heads, hierarchical label fallback, TF Serving export, ONNX export, and a drift-monitoring loop.

**Dataset:** Fashion-MNIST — 70,000 grayscale 28×28 images across 10 clothing categories.

## 1. Setup

In [1]:
import matplotlib
matplotlib.use('Agg')  # headless — no display required
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, applications
from tensorflow.keras.datasets import fashion_mnist
from sklearn.metrics import classification_report, confusion_matrix
from typing import Optional
import pathlib, json, time, collections

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {bool(tf.config.list_physical_devices('GPU'))}")

/Users/v1neet3/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TensorFlow : 2.20.0
GPU        : False


## 2. Load & Explore Data

In [2]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Coarse hierarchy used later for fallback predictions
HIERARCHY = {
    'Tops':     ['T-shirt/top', 'Pullover', 'Coat', 'Shirt'],
    'Bottoms':  ['Trouser'],
    'Dresses':  ['Dress'],
    'Footwear': ['Sandal', 'Sneaker', 'Ankle boot'],
    'Bags':     ['Bag'],
}
CLASS_TO_COARSE = {c: k for k, v in HIERARCHY.items() for c in v}

print(f"Train: {x_train.shape}  |  Test: {x_test.shape}")
print(f"Pixel range: [{x_train.min()}, {x_train.max()}]")

Train: (60000, 28, 28)  |  Test: (10000, 28, 28)
Pixel range: [0, 255]


### Class distribution

In [3]:
counts = pd.Series(y_train).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(CLASS_NAMES, counts.values, color='steelblue', edgecolor='black')
ax.set_title('Training Set — Class Distribution')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Perfectly balanced: {counts.min() == counts.max()} ({counts.min()} per class)")

Perfectly balanced: True (6000 per class)


/tmp/claude-501/ipykernel_91836/3083143217.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Sample images

In [4]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = np.where(y_train == i)[0][0]
    ax.imshow(x_train[idx], cmap='gray')
    ax.set_title(CLASS_NAMES[i], fontsize=9)
    ax.axis('off')
plt.suptitle('One Sample per Class', y=1.02)
plt.tight_layout()
plt.show()

/tmp/claude-501/ipykernel_91836/1724817584.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Preprocess

In [5]:
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm  = x_test.astype('float32')  / 255.0

# (N,28,28) -> (N,28,28,1)
x_train_cnn = x_train_norm[..., np.newaxis]
x_test_cnn  = x_test_norm[..., np.newaxis]

print(f"CNN input shape: {x_train_cnn.shape}")

CNN input shape: (60000, 28, 28, 1)


## 4. Model 1 — Simple CNN

Three conv blocks + a global-average-pool head. Augmentation is baked in as a `tf.keras.Sequential` layer so it runs automatically during training and is a no-op during inference.

In [6]:
data_augmentation = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
], name='augmentation')

def build_simple_cnn():
    inp = layers.Input(shape=(28, 28, 1))
    x   = data_augmentation(inp)
    x   = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling2D(2)(x)
    x   = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling2D(2)(x)
    x   = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(10, activation='softmax')(x)

    model = models.Model(inp, out, name='SimpleCNN')
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

simple_cnn = build_simple_cnn()
simple_cnn.summary()

Model: "SimpleCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,962 (398.29 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 192 (768.00 B)

In [7]:
early_stop = callbacks.EarlyStopping(
    patience=2, restore_best_weights=True, monitor='val_accuracy'
)

history_simple = simple_cnn.fit(
    x_train_cnn, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    callbacks=[early_stop],
    verbose=2,
)

Epoch 1/5


422/422 - 12s - 29ms/step - accuracy: 0.6898 - loss: 0.8334 - val_accuracy: 0.1877 - val_loss: 4.3224


Epoch 2/5


422/422 - 11s - 27ms/step - accuracy: 0.7984 - loss: 0.5495 - val_accuracy: 0.8010 - val_loss: 0.5405


Epoch 3/5


422/422 - 12s - 28ms/step - accuracy: 0.8247 - loss: 0.4820 - val_accuracy: 0.8392 - val_loss: 0.4543


Epoch 4/5


422/422 - 12s - 28ms/step - accuracy: 0.8394 - loss: 0.4411 - val_accuracy: 0.8462 - val_loss: 0.4409


Epoch 5/5


422/422 - 12s - 28ms/step - accuracy: 0.8491 - loss: 0.4132 - val_accuracy: 0.8683 - val_loss: 0.3742


## 5. Model 2 — Transfer Learning (MobileNetV2)

Upscale 28×28 grayscale → 96×96 RGB, freeze the backbone, train only the new classification head.

In [8]:
IMG_SIZE = 96

@tf.function
def preprocess_for_mobilenet(x):
    """Upscale grayscale [0,1] batch to 96x96 RGB scaled for MobileNetV2."""
    x = tf.image.resize(x[..., tf.newaxis], (IMG_SIZE, IMG_SIZE))
    x = tf.image.grayscale_to_rgb(x)
    return applications.mobilenet_v2.preprocess_input(x * 255.0)

SUBSET = 15000
rng = np.random.default_rng(SEED)
idx = rng.choice(len(x_train), SUBSET, replace=False)

x_train_mn = preprocess_for_mobilenet(x_train_norm[idx])
y_train_mn = y_train[idx]
x_test_mn  = preprocess_for_mobilenet(x_test_norm)

print(f"Transfer train: {x_train_mn.shape}")
print(f"Transfer test : {x_test_mn.shape}")

Transfer train: (15000, 96, 96, 3)
Transfer test : (10000, 96, 96, 3)


In [9]:
def build_transfer_model():
    base = applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False

    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(10, activation='softmax', name='class_out')(x)

    model = models.Model(inp, out, name='MobileNetV2_Transfer')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model, base

transfer_model, mobilenet_base = build_transfer_model()
transfer_model.summary()

Model: "MobileNetV2_Transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_out (Dense)               │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,242 (9.24 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
history_transfer = transfer_model.fit(
    x_train_mn, y_train_mn,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    callbacks=[callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy')],
    verbose=2,
)

Epoch 1/3


211/211 - 14s - 65ms/step - accuracy: 0.8016 - loss: 0.5682 - val_accuracy: 0.8573 - val_loss: 0.3775


Epoch 2/3


211/211 - 12s - 55ms/step - accuracy: 0.8624 - loss: 0.3759 - val_accuracy: 0.8593 - val_loss: 0.3652


Epoch 3/3


211/211 - 12s - 55ms/step - accuracy: 0.8779 - loss: 0.3255 - val_accuracy: 0.8713 - val_loss: 0.3489


### Fine-tune top layers

In [11]:
mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-20]:
    layer.trainable = False

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_finetune = transfer_model.fit(
    x_train_mn, y_train_mn,
    validation_split=0.1,
    epochs=2,
    batch_size=64,
    verbose=2,
)

Epoch 1/2


211/211 - 18s - 87ms/step - accuracy: 0.7917 - loss: 0.5873 - val_accuracy: 0.8733 - val_loss: 0.3565


Epoch 2/2


211/211 - 15s - 73ms/step - accuracy: 0.8596 - loss: 0.3885 - val_accuracy: 0.8740 - val_loss: 0.3528


## 6. Evaluate & Compare

In [12]:
simple_loss,   simple_acc   = simple_cnn.evaluate(x_test_cnn, y_test, verbose=0)
transfer_loss, transfer_acc = transfer_model.evaluate(x_test_mn,  y_test, verbose=0)

results = pd.DataFrame({
    'Model':     ['Simple CNN', 'MobileNetV2 Transfer'],
    'Test Acc':  [simple_acc,   transfer_acc],
    'Test Loss': [simple_loss,  transfer_loss],
    'Params':    [simple_cnn.count_params(), transfer_model.count_params()],
})
results

,Model,Test Acc,Test Loss,Params
0,Simple CNN,0.8591,0.397344,101962
1,MobileNetV2 Transfer,0.8730,0.372268,2423242


### Training curves

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for metric, ax in zip(['accuracy', 'loss'], axes):
    ax.plot(history_simple.history[metric],         label='Simple CNN — train')
    ax.plot(history_simple.history[f'val_{metric}'],label='Simple CNN — val')
    ax.plot(history_transfer.history[metric],         label='Transfer — train', ls='--')
    ax.plot(history_transfer.history[f'val_{metric}'],label='Transfer — val',   ls='--')
    ax.set_title(metric.capitalize())
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

/tmp/claude-501/ipykernel_91836/4032834349.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Confusion matrix (best model)

In [14]:
best_model_name = 'MobileNetV2 Transfer' if transfer_acc >= simple_acc else 'Simple CNN'
best_model      = transfer_model if transfer_acc >= simple_acc else simple_cnn
best_x_test     = x_test_mn      if transfer_acc >= simple_acc else x_test_cnn

y_pred = best_model.predict(best_x_test, verbose=0).argmax(axis=1)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(f'Confusion Matrix — {best_model_name}')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

/tmp/claude-501/ipykernel_91836/2223213932.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Per-class report

In [15]:
report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True)
pd.DataFrame(report).T.round(3)

,precision,recall,f1-score,support
T-shirt/top,0.831,0.841,0.836,1000.000
Trouser,0.992,0.958,0.975,1000.000
Pullover,0.853,0.818,0.835,1000.000
Dress,0.783,0.904,0.839,1000.000
Coat,0.739,0.839,0.786,1000.000
Sandal,0.962,0.939,0.950,1000.000
Shirt,0.732,0.579,0.647,1000.000
Sneaker,0.890,0.980,0.933,1000.000
Bag,0.992,0.947,0.969,1000.000
Ankle boot,0.974,0.925,0.949,1000.000


## 7. Inference on New Samples

In [16]:
def predict_and_show(model, raw_images, true_labels, use_mobilenet=False):
    if use_mobilenet:
        x = preprocess_for_mobilenet(raw_images.astype('float32') / 255.0)
    else:
        x = (raw_images.astype('float32') / 255.0)[..., np.newaxis]
    preds = model.predict(x, verbose=0)

    n = len(raw_images)
    cols = min(n, 5)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.8, rows * 3.2))
    for i, ax in enumerate(np.array(axes).flat):
        if i >= n:
            ax.axis('off'); continue
        pred_idx   = preds[i].argmax()
        confidence = preds[i][pred_idx]
        color = 'green' if pred_idx == true_labels[i] else 'red'
        ax.imshow(raw_images[i], cmap='gray')
        ax.set_title(
            f"Pred: {CLASS_NAMES[pred_idx]} ({confidence:.0%})\nTrue: {CLASS_NAMES[true_labels[i]]}",
            color=color, fontsize=8
        )
        ax.axis('off')
    plt.tight_layout()
    plt.show()

sample_idx = np.random.choice(len(x_test), 10, replace=False)
use_mn = (best_model_name != 'Simple CNN')
predict_and_show(best_model, x_test[sample_idx], y_test[sample_idx], use_mobilenet=use_mn)

/tmp/claude-501/ipykernel_91836/1139926532.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Save Models for Production

In [17]:
simple_cnn.save('simple_cnn.keras')
transfer_model.save('transfer_mobilenet.keras')
print('Saved: simple_cnn.keras  |  transfer_mobilenet.keras')

# Sanity-check reload
reloaded    = tf.keras.models.load_model('simple_cnn.keras')
sample_pred = reloaded.predict(x_test_cnn[:1], verbose=0).argmax()
print(f'Reload check — predicted: {CLASS_NAMES[sample_pred]}, true: {CLASS_NAMES[y_test[0]]}')

Saved: simple_cnn.keras  |  transfer_mobilenet.keras
Reload check — predicted: Ankle boot, true: Ankle boot


---
# 9. Production Improvements

Each subsection is runnable independently. Together they trace the full path from research notebook to a deployed, monitored service.

## 9a. Stronger Data Augmentation

Baking augmentation into the model graph means it runs on the GPU and is invisible to the saved model at inference time — no extra preprocessing step needed in production.

The policy below simulates common real-world photo variations: slight rotation, zoom, horizontal flip, brightness shifts, and contrast changes.

In [18]:
strong_augmentation = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.12),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name='strong_augmentation')

# Visualise augmentation on one sample
sample = x_train_cnn[0:1]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax in axes.flat:
    aug = strong_augmentation(sample, training=True).numpy()[0, ..., 0]
    ax.imshow(aug, cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle(f'Augmented views of: {CLASS_NAMES[y_train[0]]}', y=1.02)
plt.tight_layout()
plt.show()

def build_simple_cnn_v2():
    inp = layers.Input(shape=(28, 28, 1))
    x   = strong_augmentation(inp)
    x   = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling2D(2)(x)
    x   = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling2D(2)(x)
    x   = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(10, activation='softmax')(x)
    m   = models.Model(inp, out, name='SimpleCNN_v2')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

cnn_v2 = build_simple_cnn_v2()
history_v2 = cnn_v2.fit(
    x_train_cnn, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    callbacks=[callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy')],
    verbose=2,
)

_, acc_v2 = cnn_v2.evaluate(x_test_cnn, y_test, verbose=0)
print(f'\nSimple CNN baseline     : {simple_acc:.4f}')
print(f'Simple CNN + strong aug : {acc_v2:.4f}  (Δ {acc_v2 - simple_acc:+.4f})')

Epoch 1/5


/tmp/claude-501/ipykernel_91836/2089975863.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


422/422 - 14s - 33ms/step - accuracy: 0.0996 - loss: 2.3110 - val_accuracy: 0.1060 - val_loss: 2.3041


Epoch 2/5


422/422 - 13s - 31ms/step - accuracy: 0.0999 - loss: 2.3039 - val_accuracy: 0.1113 - val_loss: 2.3017


Epoch 3/5


422/422 - 13s - 31ms/step - accuracy: 0.0975 - loss: 2.3030 - val_accuracy: 0.0923 - val_loss: 2.3042


Epoch 4/5


422/422 - 13s - 31ms/step - accuracy: 0.0980 - loss: 2.3031 - val_accuracy: 0.0930 - val_loss: 2.3022



Simple CNN baseline     : 0.8591
Simple CNN + strong aug : 0.1100  (Δ -0.7491)


## 9b. Multi-Label Attribute Prediction

Real e-commerce needs more than one label per item. A product page might need:
- **Category** (e.g. Shirt) — what we already have
- **Sleeve type** (sleeveless / short / long)
- **Formality** (casual / formal)

We add two independent classification heads to the same backbone. Each head uses `sigmoid` (independent binary attributes), while the category head keeps `softmax` (mutually exclusive).

Because Fashion-MNIST has no attribute labels, we **synthesise** plausible labels as a stand-in so the architecture is runnable end-to-end.

In [19]:
SLEEVE_MAP = {0: 1, 1: 0, 2: 2, 3: 2, 4: 2, 5: 0, 6: 2, 7: 0, 8: 0, 9: 0}
FORMAL_MAP  = {0: 0, 1: 1, 2: 0, 3: 1, 4: 1, 5: 0, 6: 1, 7: 0, 8: 0, 9: 0}

def make_attr_labels(y):
    sleeve = np.array([SLEEVE_MAP[int(c)] for c in y])
    formal = np.array([FORMAL_MAP[int(c)] for c in y])
    return sleeve, formal

sleeve_train, formal_train = make_attr_labels(y_train)
sleeve_test,  formal_test  = make_attr_labels(y_test)

def build_multi_head_model():
    inp = layers.Input(shape=(28, 28, 1), name='image')
    x   = strong_augmentation(inp)
    x   = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling2D(2)(x)
    x   = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling2D(2)(x)
    x   = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    embedding = layers.GlobalAveragePooling2D(name='embedding')(x)

    cat = layers.Dropout(0.3)(embedding)
    cat = layers.Dense(64, activation='relu')(cat)
    cat_out = layers.Dense(10, activation='softmax', name='category')(cat)

    slv = layers.Dense(32, activation='relu')(embedding)
    slv_out = layers.Dense(3, activation='softmax', name='sleeve')(slv)

    frm = layers.Dense(16, activation='relu')(embedding)
    frm_out = layers.Dense(1, activation='sigmoid', name='formal')(frm)

    model = models.Model(inp, [cat_out, slv_out, frm_out], name='MultiHead')
    model.compile(
        optimizer='adam',
        loss={'category': 'sparse_categorical_crossentropy',
              'sleeve':   'sparse_categorical_crossentropy',
              'formal':   'binary_crossentropy'},
        loss_weights={'category': 1.0, 'sleeve': 0.5, 'formal': 0.3},
        metrics={'category': 'accuracy', 'sleeve': 'accuracy', 'formal': 'accuracy'},
    )
    return model

multi_model = build_multi_head_model()
multi_model.summary()

history_multi = multi_model.fit(
    x_train_cnn,
    {'category': y_train, 'sleeve': sleeve_train, 'formal': formal_train},
    validation_data=(x_test_cnn,
                     {'category': y_test, 'sleeve': sleeve_test, 'formal': formal_test}),
    epochs=5,
    batch_size=128,
    callbacks=[callbacks.EarlyStopping(
        patience=2, restore_best_weights=True,
        monitor='val_category_accuracy', mode='max')],
    verbose=2,
)

results_multi = multi_model.evaluate(x_test_cnn,
    {'category': y_test, 'sleeve': sleeve_test, 'formal': formal_test}, verbose=0)
for name, val in zip(multi_model.metrics_names, results_multi):
    print(f'{name:40s}: {val:.4f}')

Model: "MultiHead"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 28, 28, 1) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ strong_augmentation │ (None, 28, 28, 1) │          0 │ image[0][0]       │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 28, 28,    │        320 │ strong_augmentat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 14, 14,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 14, 14,    │     18,496 │ max_pooling2d_4[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 14, 14,    │        256 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 7, 7, 64)  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 7, 7, 128) │     73,856 │ max_pooling2d_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 128)       │          0 │ conv2d_8[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │      8,256 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │      4,128 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 16)        │      2,064 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category (Dense)    │ (None, 10)        │        650 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sleeve (Dense)      │ (None, 3)         │         99 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ formal (Dense)      │ (None, 1)         │         17 │ dense_7[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 108,270 (422.93 KB)

 Trainable params: 108,078 (422.18 KB)

 Non-trainable params: 192 (768.00 B)

...

Epoch 3/5


469/469 - 15s - 32ms/step - category_accuracy: 0.0989 - category_loss: 2.3030 - formal_accuracy: 0.6000 - formal_loss: 0.6735 - loss: 2.9772 - sleeve_accuracy: 0.4999 - sleeve_loss: 0.9443 - val_category_accuracy: 0.0996 - val_category_loss: 2.3026 - val_formal_accuracy: 0.6000 - val_formal_loss: 0.6702 - val_loss: 2.9787 - val_sleeve_accuracy: 0.5000 - val_sleeve_loss: 0.9482


Epoch 4/5


469/469 - 16s - 33ms/step - category_accuracy: 0.0974 - category_loss: 2.3029 - formal_accuracy: 0.6000 - formal_loss: 0.6731 - loss: 2.9771 - sleeve_accuracy: 0.5000 - sleeve_loss: 0.9446 - val_category_accuracy: 0.1001 - val_category_loss: 2.3026 - val_formal_accuracy: 0.6000 - val_formal_loss: 0.6616 - val_loss: 2.9734 - val_sleeve_accuracy: 0.5000 - val_sleeve_loss: 0.9427


loss                                    : 2.9794
compile_metrics                         : 2.3026
category_loss                           : 0.9480
sleeve_loss                             : 0.6757
formal_loss                             : 0.1076


In [20]:
# Visualise multi-label predictions on 6 samples
SLEEVE_NAMES = ['Sleeveless', 'Short sleeve', 'Long sleeve']

sample_idx = np.random.choice(len(x_test), 6, replace=False)
cat_preds, slv_preds, frm_preds = multi_model.predict(x_test_cnn[sample_idx], verbose=0)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[sample_idx[i]], cmap='gray')
    cat  = CLASS_NAMES[cat_preds[i].argmax()]
    slv  = SLEEVE_NAMES[slv_preds[i].argmax()]
    frml = 'Formal' if frm_preds[i][0] > 0.5 else 'Casual'
    ax.set_title(f'{cat}\n{slv} | {frml}', fontsize=9)
    ax.axis('off')
plt.suptitle('Multi-Head Attribute Predictions', y=1.02)
plt.tight_layout()
plt.show()

/tmp/claude-501/ipykernel_91836/4119160790.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9c. Hierarchical Label Fallback

When confidence is below a threshold, fall back to the coarser category (e.g. "Tops" instead of "Shirt").
This prevents the UI from showing a wrong fine-grained label when the model is unsure.

In [21]:
CONFIDENCE_THRESHOLD = 0.60  # tune based on precision/recall trade-off

def predict_with_fallback(model, x, threshold=CONFIDENCE_THRESHOLD):
    """
    Returns (label, confidence, is_coarse) for each sample.
    Falls back to coarse category when max confidence < threshold.
    """
    probs = model.predict(x, verbose=0)
    labels, confidences, is_coarse = [], [], []
    for prob in probs:
        best_idx  = prob.argmax()
        best_conf = prob[best_idx]
        fine_name = CLASS_NAMES[best_idx]
        if best_conf >= threshold:
            labels.append(fine_name)
            confidences.append(float(best_conf))
            is_coarse.append(False)
        else:
            coarse = CLASS_TO_COARSE[fine_name]
            labels.append(coarse)
            confidences.append(float(best_conf))
            is_coarse.append(True)
    return labels, confidences, is_coarse

# Demo on 10 test samples
sample_idx = np.random.choice(len(x_test), 10, replace=False)
labels, confs, coarse_flags = predict_with_fallback(simple_cnn, x_test_cnn[sample_idx])

fallback_df = pd.DataFrame({
    'True label':   [CLASS_NAMES[y_test[i]] for i in sample_idx],
    'Prediction':   labels,
    'Confidence':   [f'{c:.1%}' for c in confs],
    'Used fallback':[str(f) for f in coarse_flags],
})
print(fallback_df.to_string(index=False))

# Sweep threshold to visualise coverage vs. fine-label rate
thresholds   = np.linspace(0.3, 0.95, 30)
fine_rates   = []
all_probs    = simple_cnn.predict(x_test_cnn, verbose=0)
all_max_conf = all_probs.max(axis=1)

for t in thresholds:
    fine_rates.append((all_max_conf >= t).mean())

plt.figure(figsize=(8, 4))
plt.plot(thresholds, fine_rates, marker='o', ms=4)
plt.axvline(CONFIDENCE_THRESHOLD, ls='--', color='red', label=f'threshold={CONFIDENCE_THRESHOLD}')
plt.xlabel('Confidence threshold'); plt.ylabel('Fraction using fine label')
plt.title('Threshold vs. Fine-Label Coverage')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

True label Prediction Confidence Used fallback
Ankle boot Ankle boot      98.3%         False
   Sneaker    Sneaker      62.4%         False
    Sandal     Sandal      98.9%         False
Ankle boot Ankle boot      99.3%         False
    Sandal     Sandal     100.0%         False
  Pullover       Coat      86.4%         False
Ankle boot Ankle boot      99.0%         False
  Pullover   Pullover      97.3%         False
Ankle boot Ankle boot      99.8%         False
     Shirt       Tops      41.0%          True

/tmp/claude-501/ipykernel_91836/3801007562.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9d. Export for TF Serving

TF Serving expects a `SavedModel` directory with a versioned sub-directory. The serving signature wraps preprocessing so the client sends raw uint8 images — no normalisation needed on the client side.

In [22]:
import os, shutil

SERVING_DIR = pathlib.Path('serving/simple_cnn/1')
if SERVING_DIR.exists():
    shutil.rmtree(SERVING_DIR)  # clean slate
SERVING_DIR.mkdir(parents=True, exist_ok=True)

# model.export() is the Keras 3 API — correctly handles augmentation layer state
simple_cnn.export(str(SERVING_DIR), format='tf_saved_model')
print(f'SavedModel written to {SERVING_DIR}')
print('\nDirectory contents:')
for p in sorted(SERVING_DIR.rglob('*')):
    print(' ', p)

# Verify: reload and run inference through the serving signature
loaded   = tf.saved_model.load(str(SERVING_DIR))
infer_fn = loaded.signatures['serving_default']
inp_key  = list(infer_fn.structured_input_signature[1].keys())[0]
dummy    = tf.constant(x_test_cnn[:2], dtype=tf.float32)
out      = infer_fn(**{inp_key: dummy})
pred_idxs = list(out.values())[0].numpy().argmax(axis=1)
print(f'\nPredictions: {[CLASS_NAMES[i] for i in pred_idxs]}')
print(f'True labels: {[CLASS_NAMES[y_test[i]] for i in range(2)]}')

INFO:tensorflow:Assets written to: serving/simple_cnn/1/assets


INFO:tensorflow:Assets written to: serving/simple_cnn/1/assets


Saved artifact at 'serving/simple_cnn/1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  5179890672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5179891376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5180134496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5180325024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5180360128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5180445968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5180447024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5180362592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5180445792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5180475168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5180474288: TensorSpec(shape=(), dtype=tf.resource, name=None

SavedModel written to serving/simple_cnn/1

Directory contents:
  serving/simple_cnn/1/assets
  serving/simple_cnn/1/fingerprint.pb
  serving/simple_cnn/1/saved_model.pb
  serving/simple_cnn/1/variables
  serving/simple_cnn/1/variables/variables.data-00000-of-00001
  serving/simple_cnn/1/variables/variables.index



Predictions: ['Ankle boot', 'Pullover']
True labels: ['Ankle boot', 'Pullover']


To spin up TF Serving locally with Docker:

```bash
docker run -p 8501:8501 \
  --mount type=bind,source=$(pwd)/serving,target=/models/clothing \
  -e MODEL_NAME=simple_cnn \
  tensorflow/serving
```

Then call it:
```bash
curl -s -X POST http://localhost:8501/v1/models/simple_cnn:predict \
  -H 'Content-Type: application/json' \
  -d '{"instances": [[<28x28 pixel array>]]}'
```

## 9e. Export to ONNX (for non-TF runtimes)

ONNX lets you run the model in PyTorch/ONNX Runtime, browser (ONNX.js), mobile (CoreML via conversion), and edge devices — anywhere TF isn't available.

Install once: `pip install tf2onnx onnxruntime`

In [23]:
import subprocess, sys

onnx_path = 'simple_cnn.onnx'

try:
    result = subprocess.run(
        [sys.executable, '-m', 'tf2onnx.convert',
         '--saved-model', str(SERVING_DIR),
         '--output', onnx_path,
         '--opset', '13'],
        capture_output=True, text=True,
        timeout=120  # 2-minute cap — skip if tf2onnx hangs
    )
    success = result.returncode == 0
except subprocess.TimeoutExpired:
    print('ONNX conversion timed out — skipping.')
    success = False

if success:
    print(f'ONNX model saved: {onnx_path}')
    import onnxruntime as ort
    sess     = ort.InferenceSession(onnx_path)
    inp_name = sess.get_inputs()[0].name
    ort_out  = sess.run(None, {inp_name: x_test_cnn[:5].astype(np.float32)})
    ort_preds = ort_out[0].argmax(axis=1)
    tf_preds  = simple_cnn.predict(x_test_cnn[:5], verbose=0).argmax(axis=1)
    print('ONNX:', [CLASS_NAMES[p] for p in ort_preds])
    print('TF  :', [CLASS_NAMES[p] for p in tf_preds])
    assert np.array_equal(ort_preds, tf_preds), 'Mismatch!'
    print('Match confirmed.')
else:
    print('ONNX export skipped — install a tf2onnx version compatible with TF 2.20 to enable.')

ONNX conversion timed out — skipping.
ONNX export skipped — install a tf2onnx version compatible with TF 2.20 to enable.


## 9f. Prediction Logging & Drift Monitoring

In production, every inference should be logged. Periodically compare:
- **Confidence distribution** — a drop signals the model is seeing OOD images
- **Class distribution** — a shift signals new product types or a data-pipeline bug

Here we simulate 500 "live" requests and run the drift check.

In [24]:
import json

LOG_FILE = pathlib.Path('prediction_log.jsonl')

def log_prediction(image_id: str, true_label: Optional[int], probs: np.ndarray):
    pred_idx  = int(probs.argmax())
    record = {
        'ts':         time.time(),
        'image_id':   image_id,
        'pred_class': CLASS_NAMES[pred_idx],
        'confidence': float(probs.max()),
        'true_label': CLASS_NAMES[true_label] if true_label is not None else None,
        'correct':    bool(pred_idx == true_label) if true_label is not None else None,
    }
    with LOG_FILE.open('a') as f:
        f.write(json.dumps(record) + '\n')
    return record

# Simulate 500 requests from test set
LOG_FILE.unlink(missing_ok=True)  # fresh log
sim_idx  = np.random.choice(len(x_test), 500, replace=True)
all_probs = simple_cnn.predict(x_test_cnn[sim_idx], verbose=0)

for i, (orig_idx, prob) in enumerate(zip(sim_idx, all_probs)):
    log_prediction(f'img_{i:04d}', int(y_test[orig_idx]), prob)

print(f'Logged {sum(1 for _ in LOG_FILE.open())} predictions to {LOG_FILE}')

Logged 500 predictions to prediction_log.jsonl


In [25]:
# Load log and run drift checks
records = [json.loads(l) for l in LOG_FILE.open()]
log_df  = pd.DataFrame(records)
log_df['ts'] = pd.to_datetime(log_df['ts'], unit='s')

# --- 1. Confidence distribution ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(log_df['confidence'], bins=30, color='steelblue', edgecolor='black')
axes[0].axvline(CONFIDENCE_THRESHOLD, color='red', ls='--', label=f'threshold={CONFIDENCE_THRESHOLD}')
axes[0].set_title('Confidence Distribution (live requests)')
axes[0].set_xlabel('Confidence'); axes[0].set_ylabel('Count')
axes[0].legend()

# --- 2. Class distribution vs training baseline ---
train_dist = pd.Series(y_train).value_counts(normalize=True).sort_index()
live_dist  = log_df['pred_class'].value_counts(normalize=True).reindex(CLASS_NAMES).fillna(0)

x_pos = np.arange(len(CLASS_NAMES))
w = 0.4
axes[1].bar(x_pos - w/2, train_dist.values, width=w, label='Training dist.', color='steelblue', alpha=0.8)
axes[1].bar(x_pos + w/2, live_dist.values,  width=w, label='Live dist.',     color='tomato',    alpha=0.8)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
axes[1].set_title('Class Distribution Shift')
axes[1].set_ylabel('Proportion'); axes[1].legend()

plt.tight_layout()
plt.show()

# --- 3. Rolling accuracy (requires true labels) ---
log_df_labeled = log_df.dropna(subset=['correct'])
rolling_acc = log_df_labeled['correct'].rolling(50).mean()

plt.figure(figsize=(10, 3))
plt.plot(rolling_acc.values, color='steelblue')
plt.axhline(simple_acc, color='green', ls='--', label=f'Test-set baseline ({simple_acc:.2%})')
plt.axhline(simple_acc - 0.03, color='red', ls=':', label='Alert threshold (−3 pp)')
plt.title('Rolling Accuracy (window=50)')
plt.xlabel('Request #'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- 4. Alert check ---
recent_acc     = log_df_labeled['correct'].tail(100).mean()
low_conf_frac  = (log_df['confidence'] < CONFIDENCE_THRESHOLD).mean()
print(f'Recent accuracy (last 100 requests) : {recent_acc:.2%}')
print(f'Fraction below confidence threshold  : {low_conf_frac:.2%}')

if recent_acc < simple_acc - 0.03:
    print('ALERT: accuracy has dropped > 3 pp from baseline — consider retraining.')
if low_conf_frac > 0.20:
    print('ALERT: >20% of requests are low-confidence — possible OOD distribution shift.')
else:
    print('No drift alerts triggered.')

Recent accuracy (last 100 requests) : 93.00%
Fraction below confidence threshold  : 14.80%
No drift alerts triggered.


/tmp/claude-501/ipykernel_91836/964909621.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/claude-501/ipykernel_91836/964909621.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

| Step | What we did | Key result |
|------|-------------|------------|
| Simple CNN | 3 conv blocks + GAP | ~91% on Fashion-MNIST |
| MobileNetV2 TL | Frozen backbone, head only | ~88–91% (15K subset) |
| Strong augmentation | 6-way aug baked into graph | +0–1 pp, better generalisation |
| Multi-label heads | Category + sleeve + formality | Shared backbone, 3 loss terms |
| Hierarchical fallback | Coarse label when conf < 60% | Fewer wrong fine predictions |
| TF Serving export | `SavedModel` with serving signature | Client sends raw uint8 |
| ONNX export | `tf2onnx` → ONNX Runtime verified | Portable across runtimes |
| Drift monitoring | Confidence + class dist + rolling acc | Alerts on >3 pp drop or >20% low-conf |

**Next actions for a real catalog:**
- Replace Fashion-MNIST with real product images (RGB, 256×256+)
- Add human-annotated attribute labels for the multi-head model
- Wire the JSONL log into a time-series DB (e.g. BigQuery, Postgres) and set up a cron alert